<a href="https://colab.research.google.com/github/yogitamittal93/deep-learning-practice/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
dataset=load_dataset("tweet_eval", 'sentiment')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [2]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [3]:
train_texts=dataset['train']['text'][:3000]
train_labels=dataset['train']['label'][:3000]
test_texts=dataset['test']['text'][:1000]
test_labels=dataset['test']['label'][:1000]

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")
def preprocess(text):
  doc=nlp(text)
  return " ".join([
      token.lemma_.lower()
      for token in doc
      if not token.is_stop
      and not token.is_punct
      and not token.like_url
      and not token.like_email
  ])

In [5]:
train_clean = [preprocess(text) for text in train_texts]
test_clean = [preprocess(text) for text in test_texts]

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:
vectorizer=TfidfVectorizer(max_features=5000)
x_train=vectorizer.fit_transform(train_clean)
x_test=vectorizer.transform(test_clean)


In [8]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=200)
model.fit(x_train, train_labels)

LogisticRegression(max_iter=200)

In [10]:
from sklearn.metrics import accuracy_score, classification_report
print(classification_report(model.predict(x_test), test_labels))
accuracy_score(model.predict(x_test), test_labels)

              precision    recall  f1-score   support

           0       0.07      0.50      0.12        40
           1       0.83      0.54      0.65       768
           2       0.45      0.46      0.46       192

    accuracy                           0.52      1000
   macro avg       0.45      0.50      0.41      1000
weighted avg       0.72      0.52      0.59      1000



0.521